# NLP Masterclass 01: Text Preprocessing & Tokenization

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** Python fundamentals, Basic Probability  
**Difficulty:** ⭐ Beginner → Intermediate  
**Estimated Time:** 90–120 minutes

---

## 🎓 Socratic Deep Check

> *"GPT-4 has a vocabulary of ~100,256 tokens. Why is this number so specific? If we used raw characters, our sequences would be 5x longer. If we used whole words, our vocabulary would be 1,000,000+ words. How do we find the 'Goldilocks' zone between computational efficiency and linguistic expressive power?"*

---

## Why This Matters for AI Engineers

Every LLM interaction starts with tokenization. **API costs are per-token.** Context windows are measured in tokens. RAG chunking (NB21) depends on token boundaries. Even 'traditional' AI tasks like sentiment analysis or classification often fail not because the model is weak, but because the preprocessing was noisy or inconsistent.

### Learning Objectives
1. Master **Traditional Preprocessing** (Regex, Unicode, Normalization).
2. Implement **Classical Statistical Representations** (BoW, TF-IDF) from scratch.
3. Solve the **OOV (Out-of-Vocabulary)** problem with modern **Subword Algorithms** (BPE, WordPiece, Unigram).
4. Train a custom **Production-Grade Tokenizer** using HuggingFace's `tokenizers` library.

---

## 0 · Setup & Prerequisites

In [ ]:
# 1. Install necessary libraries
!pip install -q nltk spacy scikit-learn transformers tokenizers tiktoken

# 2. Download NLTK data and spaCy model
import nltk
import spacy
import subprocess

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print("✅ Setup Complete: Dependencies installed and models downloaded.")

---
## 1 · Traditional Preprocessing (The Pre-LLM Era)

Before we look at learned tokenization, we must handle the raw noise in human text. While modern LLMs are robust, traditional ML and search engines (ElasticSearch/OpenSearch) still rely heavily on these deterministic filters.

### 1.1 · The Production-Grade Cleaning Pipeline
We use **Unicode Normalization** to handle different encodings of the same character (e.g., 'e' + '´' vs 'é').

In [ ]:
import re
import unicodedata

def production_clean(text, aggressive=False):
    """
    A robust cleaning function for production pipelines.
    Normalization Form Compatibility Decomposition (NFKD) separates base chars from accents.
    """
    # 1. Unicode Normalization
    text = unicodedata.normalize('NFKD', text)
    
    # 2. Lowercasing (Standardize casing)
    text = text.lower()
    
    # 3. Strip HTML Tags (Regex-based)
    text = re.sub(r'<[^>]*>', ' ', text)
    
    # 4. Handle Emojis & Weird Punctuation (Optional)
    if aggressive:
        # Remove everything except alphanumeric and spaces
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 5. Collapse Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# TEST IT
raw_input = "  <p>The résumé was <b>PERFECT</b>! 🎉  \n Contact: me@example.com </p> "
print(f"Raw: {raw_input}")
print(f"Cleaned (Balanced):  {production_clean(raw_input)}")
print(f"Cleaned (Aggressive): {production_clean(raw_input, aggressive=True)}")

### 1.2 · Stemming vs. Lemmatization

| Feature | Stemming (e.g., Porter) | Lemmatization (e.g., spaCy) |
|---------|-------------------------|---------------------------|
| **Logic** | Hops off suffixes (Heuristic) | Uses dictionary & context (Morphological) |
| **Speed** | 🚀 Extremely Fast | 🐢 Slower (requires POS tagging) |
| **Accuracy**| Low ("studies" → "studi") | High ("studies" → "study") |
| **Usecase** | High-speed indexing, search | Complex NLP, Chatbots |

In [ ]:
from nltk.stem import PorterStemmer
import spacy

stemmer = PorterStemmer()
nlp = spacy.load("en_core_web_sm")

words = ["changing", "changes", "changed", "charger", "charge"]

print(f"{ 'Word':<12} | { 'Stemming':<12} | { 'Lemmatization':<12}")
print("-" * 45)
for w in words:
    stem = stemmer.stem(w)
    # In spaCy, we process the word through the pipeline to get the lemma
    lemma = nlp(w)[0].lemma_
    print(f"{w:<12} | {stem:<12} | {lemma:<12}")

---
## 2 · Classical Statistical NLP (The Vector Space Era)

How do we turn words into numbers without deep learning? For decades, **TF-IDF** was the gold standard for information retrieval.

### 2.1 · Bag-of-Words (BoW)
A simple count of word frequencies. 
**Problem:** It treats "the" (very common, low info) as more important than "quantum" (rare, high info).

### 2.2 · TF-IDF: The Math

$$TF(t, d) = \frac{\text{Count of term } t \text{ in doc } d}{\text{Total words in doc } d}$$

$$IDF(t, D) = \log \left( \frac{\text{Total docs in } D}{1 + \text{Docs containing } t} \right)$$

$$TF\text{-}IDF = TF \times IDF$$

Why the log? It dampens the effect of frequency, ensuring that having a term 100 times isn't 100x more important than having it once.

In [ ]:
import math
from collections import Counter
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat played with the dog"
]

def calculate_tfidf_scratch(documents):
    # 1. Collect all terms
    all_words = set(" ".join(documents).split())
    N = len(documents)
    
    # 2. IDF calculation
    idf = {}
    for word in all_words:
        doc_count = sum(1 for d in documents if word in d.split())
        idf[word] = math.log(N / (doc_count))
    
    # 3. TF-IDF Calculation for the first document
    doc0 = documents[0].split()
    counts = Counter(doc0)
    results = {}
    for word in doc0:
        tf = counts[word] / len(doc0)
        results[word] = tf * idf[word]
    return results

print("TF-IDF (Scratch) for Doc 0:")
print(calculate_tfidf_scratch(corpus))

# The Scikit-Learn Way (Optimized and Standardized)
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
print("\nSklearn Vocabulary Size:", len(vectorizer.get_feature_names_out()))

### 🎓 Concept Bridge: The Sparsity Problem

**The Senior Perspective:** Imagine a vocabulary of 50,000 words. Every sentence in your dataset is now represented as a vector of length 50,000. 
*   99.9% of that vector will be zeros. 
*   This is **The Curse of Dimensionality**. 
*   Models struggle to learn patterns when the input space is so sparse.

**This is precisely why we transitioned to Embeddings (Dense Vectors) and Subword Tokenization.**

---
## 3 · Modern Subword Tokenization

Why didn't word-level tokenization survive the LLM era?
1. **Vocabulary Bloom:** Can't store 1M+ words in GPU memory.
2. **OOV (Out-of-Vocabulary):** If the model sees 'Unbelievable' but only trained on 'Believable', it's stuck.

**Solution:** Break words into meaningful sub-units.

### 3.1 · Byte-Pair Encoding (BPE)
*Used by: GPT-2, GPT-3, GPT-4, Llama 2/3.*

**Logic:**
1. Split words into individual bytes/characters.
2. Count the most frequent adjacent pair (e.g., 't' + 'h').
3. Merge them into a single token ('th').
4. Repeat until you reach your target `vocab_size`.

In [ ]:
from collections import Counter
import re

def get_stats(vocab):
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair, v_in):
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Initial Vocab (words split into chars space-separated)
vocab = {'l o w </w>' : 5, 'l o w e r </w>' : 2, 'n e w e s t </w>': 6, 'w i d e s t </w>': 3}
num_merges = 5

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs: break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge {i+1}: {best} -> freq={pairs[best]}")
    print(f"Current state: {vocab}\n")

### 3.2 · WordPiece vs. Unigram (SentencePiece)

#### WordPiece (BERT, DistilBERT)
Similar to BPE but with a twist: instead of most frequent pair, it picks the pair that **maximizes likelihood**. 
$$Score = \frac{P(AB)}{P(A)P(B)}$$
Essentially: "Does combining A and B tell us something and reduce surprise more than keeping them separate?"

#### Unigram / SentencePiece (T5, ALBERT, Llama wrappers)
It works in reverse! It starts with a HUGE vocabulary and **removes** pieces that don't contribute much to the probability of the corpus.

---
## 4 · Production Hands-on: Building a Tokenizer

In industry, we use the `tokenizers` library (written in Rust) because performance is critical. Training a BPE tokenizer on a 1GB dataset shouldn't take hours.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Initialize an empty BPE model
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. Trainer Config
trainer = BpeTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"], 
                    vocab_size=1000) # Small for demo

# 3. Dummy dataset
data = [
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is the foundation of NLP engineering.",
    "Modern subword algorithms handle out-of-vocabulary words intelligently."
]

# 4. Train
tokenizer.train_from_iterator(data, trainer)

# 5. Test it!
output = tokenizer.encode("The foundation of NLP is tokenization.")
print(f"Tokens: {output.tokens}")
print(f"IDs:    {output.ids}")

# SAVE
tokenizer.save("custom_tokenizer.json")

---
## ✅ Knowledge Check

### Q1: The OOV Paradox
Which algorithm handles the word "Hyper-loop" better if it was never in the training set: **Word-level** or **Subword (BPE)**?

<details><summary>👉 Answer</summary>
Subword (BPE). Word-level would flag it as [UNK] (Unknown). BPE would likely break it into ["Hyper", "-", "loop"], allowing the model to aggregate the meanings of those pieces.
</details>

### Q2: Mathematical Efficiency
Why does TF-IDF use the Logarithm for the Inverse Document Frequency? 

<details><summary>👉 Answer</summary>
Without the log, IDF values would grow linearly with the number of documents. For a corpus of 1 million docs where a word appears in only 1, the multiplier would be 1,000,000. This would drastically over-emphasize rare words, making them noise. The log dampens this growth, mapping the IDF range to something manageable and proportional.
</details>

### Q3: Cost Management (Senior Level)
You are optimizing an API-based system using GPT-4. You notice that the prompt "Tell me about AI" costs 5 tokens, but " Tell me about AI" (with a leading space) also costs 5 tokens but has a slightly different combined ID. Why are spaces often bundled into the word token in BPE?

<details><summary>👉 Answer</summary>
BPE (specifically the version used by OpenAI) treats spaces as part of the string. By merging spaces into the word (e.g., "_the" instead of [" ", "the"]), we effectively halve the sequence length for the model. Shorter sequences = faster inference and lower API costs.
</details>

---

## 🔨 Practical Challenge

**Challenge:** Compare the encoding of the following sentence using **Tiktoken (GPT-4)** and the **Llama 3** tokenizer (auto-loading from HF). 
*   Sentence: "AI Engineering is the most valuable skill of 2026!"
*   Which model produces fewer tokens? Why?

--- 
**Next →** [NLP 02: Word Embeddings & Word2Vec](02_word_embeddings_and_word2vec.ipynb)